## Abstractive Summarization Project
This project explores different types of models for abstractive text summarization:

- **BERT** (Encoder-only model)
- **BART** (Encoder-Decoder model)
- **T5** (Encoder-Decoder model)
- **PEGASUS** (Encoder-Decoder model)
## Summary
The aim of this project, is to fine tune state of the art Language Models, mentioned above.

I successfully fine-tuned the **T5-small** model. You will find an evaluation.png file in this repository which will show a rough graph for the fine-tuned T5-small model. However, due to limited GPU resources, I was unable to fine-tune **BART** and** PEGASUS**. Further details will be provided during the presentation.

>**For the Models Section:** Please look into the T5 Section first because it contains a detailed explanation of each of the function for fine tuning a Language Model , which are also used in the BART and PEGASUS section.

For a more detailed analysis and an explanation of the algorithms used, please refer to the **README**

---

## Notebook Structure

This notebook consists of three main sections:

1. **Library Installation**

2. **Dataset Preparation**

3. **Models**

## Running Instructions

You must run the **Library Installation** and **Dataset Preparation** sections before proceeding to the **Models** section.

Each **Model** section is independent and contains all the necessary steps for fine-tuning a Language Model.

## Fine-Tuning Steps

**Each Model section includes the following steps:**

1. Load the dataset and split it.

2. Load the tokenizer to preprocess the dataset.

3. Define the evaluation metrics.

4. Load the Transformer Model.

5. Define the training arguments and Trainer.

6. Train the model.

7. Push the trained model to Hugging Face.

By following these structured steps, you can effectively fine-tune and evaluate different abstractive summarization models.

## 1. Library Installation

In [ ]:
!pip install transformers datasets evaluate rouge_score

You must enter your `hugging_face api key` over here

In [1]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Importing Dataset

### CNN Daily Mail Datatset

Divide The Dataset into Trainining, Testing and Validation

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset

# Load CNN/DailyMail dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Split original 'train' into train/validation/test (80/10/10)
train_testvalid = dataset['train'].train_test_split(test_size=0.8, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Create final DatasetDict
dataset = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
import datasets
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=5):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [ ]:
show_random_elements(dataset["train"], num_examples=1)

,article,highlights,id
0,"By . Sara Malm . PUBLISHED: . 20:59 EST, 15 March 2013 . | . UPDATED: . 21:01 EST, 15 March 2013 . Although humans come in all shapes and size, the lifeless plastic dolls used to advertise the clothes we wear rarely do. So when a photograph of a fuller-figured mannequin emerged on Facebook earlier this week, the internet gave a standing ovation. The photograph, taken at a Swedish department store, shows a 'larger-than-average' clothing doll wearing a purple lingerie set, next to one of ‘normal-size’. 'Real-life sized': The photo of the Swedish mannequin was posted on Facebook earlier this week and has been shared more than 17,200 times . The photo was posted on Facebook page Women’s Rights News earlier this week with the caption: ‘Store mannequins in Sweden. They look like real women. The US should invest in some of these’ Since Tuesday the photo of the Swedish mannequin has received more than 56,000 likes. It has been shared over 17,200 times and nearly 3,000 people have left comments, a majority of which are raving about the Scandinavian retailer’s choice to use a mannequin the size of a ‘real woman’. The original poster, 29-year-old Rebecka from Malmo, found the dummy in department store Åhléns in 2010 and wrote about her discovery on her blog. The norm: Mannequins in the petite shape we are used to, at the launch of Versace's collection for H&M . ‘The mannequin to the right actually . reminds you of the size of a real human being. So nice! She is still . skinny, but it looks healthy.’ She also points a finger at Swedish retail success H&M, calling their mannequins ‘insanely skinny’ adding that the High Street giant’s dolls has ‘a waist like my shin’. H&M is not the only retailer to be critiqued for their skinny mannequins. In 2011 Gap launched their ‘always skinny’ campaign to promote a new jeans cut, accompanied by dummies with stick-thin legs. The campaign was followed by a huge online backlash where the high street brand was accused of promoting anorexia.","Picture of mannequin that 'looks like a real woman' goes viral .\nDoll larger than the average dummy found at Swedish department store .\nFacebook post of lingerie mannequin amass 56,000 likes in four days .",f399b810c1cbc8a333f05459f1e6e4a5965252a5


In [ ]:
# 3. Preprocessing: Tokenize the articles (inputs) and highlights (targets)
def preprocess_function(examples):
    inputs = tokenizer(examples["article"], max_length=512, truncation=True, padding="max_length")
    targets = tokenizer(text_target= examples["highlights"], max_length=128, truncation=True, padding="max_length")

    inputs["labels"] = targets["input_ids"]  # Assign target labels for training
    return inputs

## Evaluation

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is a widely used metric for evaluating text summarization models by comparing generated summaries with reference summaries. It measures the overlap of words or phrases, focusing on recall, precision, and F1-score.

ROUGE Variants:
= ROUGE-1: Measures the overlap of individual unigrams (single words) between the generated and reference summaries.
- ROUGE-2: Measures the overlap of bigrams (two-word sequences) to capture phrase-level similarity.
- ROUGE-L: Uses longest common subsequence (LCS) to evaluate fluency and sentence structure similarity.
- ROUGE-Sum: Specifically designed for evaluating summaries, considering multiple sentences and their relationship.

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
rouge

Creating an Evaluation Metric which will be used in the Trainer (see the training step of each of the model) to evaluate the model on **ROUGE SCORE**.

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# 3. Models

## BERT

BERT **cannot be used for Summarization**, however the concept of MLM (Masked Language Modeliing) is utilized in PEGASUS Model, therefore it was worth investing some time and looking into how the preprocessing is done.

> Note: The Presentation goes into detail on the specifics of the preprocessing of the model, truncation, padding and attention masks.

#### Preprocessing

In [ ]:
from transformers import BertTokenizer
from datasets import load_dataset

# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Load sample data
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:2]")

def visualize_preprocessing(example):
    print("\n" + "="*50)
    print("ORIGINAL ARTICLE:")
    print(example["article"][:500] + "...")  # Show first 500 characters

    print("\n" + "-"*50)
    print("ORIGINAL HIGHLIGHTS:")
    print(example["highlights"])

    # Tokenization process
    print("\n" + "-"*50)
    print("TOKENIZATION STEP:")
    tokens = tokenizer.tokenize(example["article"])
    print(f"Raw tokens count: {len(tokens)}")
    print(f"Sample tokens: {tokens[:50]}...")

    # Full processing with special tokens
    print("\n" + "-"*50)
    print("FULL PROCESSING:")
    processed = tokenizer(
        example["article"],
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
        return_attention_mask=True,
        return_token_type_ids=False
    )

    print("\nInput IDs Shape:", processed["input_ids"].shape)
    print("Attention Mask Shape:", processed["attention_mask"].shape)

    # Decode back to text with special tokens
    decoded_with_specials = tokenizer.decode(processed["input_ids"][0], skip_special_tokens=False)
    print("\nDecoded with special tokens:")
    print(decoded_with_specials[:500] + "...")

    # Show padding visualization
    print("\n" + "-"*50)
    print("PADDING VISUALIZATION:")
    input_ids = processed["input_ids"][0].tolist()
    padding_start = input_ids.index(tokenizer.pad_token_id) if tokenizer.pad_token_id in input_ids else len(input_ids)
    print(f"Padding starts at position: {padding_start}")
    print(f"Real tokens: {input_ids[:5]}...{input_ids[padding_start-5:padding_start]}")
    print(f"Padding tokens: {input_ids[padding_start:padding_start+5]}...")

    return processed

# Process and visualize first example
sample = dataset[0]
processed_sample = visualize_preprocessing(sample)

# Additional comparison: Short sentence visualization
test_sentence = "This is a test sentence to show tokenization."
print("\n" + "="*50)
print("SHORT SENTENCE ANALYSIS:")
print("Original:", test_sentence)
print("Tokenized:", tokenizer.tokenize(test_sentence))
print("Encoded:", tokenizer.encode(test_sentence, add_special_tokens=True))
print("Decoded:", tokenizer.decode(tokenizer.encode(test_sentence)))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]


ORIGINAL ARTICLE:
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s...

--------------------------------------------------
ORIGINAL HIGHLIGHTS:
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .

--------------------------------------------------
TOKENIZATION STEP:
Raw tokens count: 585
Sample tokens: ['london', ',', 'england', '(', 'reuters', ')', '-', '-', 'harry', 'potter', '

## T5 Model

Before we can feed those texts to our model, we need to preprocess them. This is done by a 🤗 Transformers `Tokenizer` which will (as the name indicates) tokenize the inputs (including converting the tokens to their corresponding IDs in the pretrained vocabulary) and put it in a format the model expects, as well as generate the other inputs that the model requires.

To do all of this, we instantiate our tokenizer with the `AutoTokenizer.from_pretrained` method, which will ensure:

- we get a tokenizer that corresponds to the model architecture we want to use,
- we download the vocabulary used when pretraining this specific checkpoint.

That vocabulary will be cached, so it's not downloaded again the next time we run the cell.

By default, the call below will use one of the fast tokenizers (backed by Rust) from the 🤗 Tokenizers library.

In [ ]:
from transformers import AutoTokenizer

checkpoint = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Depending on the model you selected, you will see different keys in the dictionary returned by the cell above. They don't matter much for what we're doing here (just know they are required by the model we will instantiate later), you can learn more about them in [this tutorial](https://huggingface.co/transformers/preprocessing.html) if you're interested.

Instead of one sentence, we can pass along a list of sentences:

In [ ]:
tokenizer("Hello, this one sentence!")

{'input_ids': [8774, 6, 48, 80, 7142, 55, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

To prepare the targets for our model, we need to tokenize them using the `text_target` parameter. This will make sure the tokenizer uses the special tokens corresponding to the targets:

In [ ]:
print(tokenizer(text_target=["Hello, this one sentence!", "This is another sentence."]))

{'input_ids': [[8774, 6, 48, 80, 7142, 55, 1], [100, 19, 430, 7142, 5, 1]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}


We can then write the function that will preprocess our samples. We just feed them to the `tokenizer` with the argument `truncation=True`. This will ensure that an input longer that what the model selected can handle will be truncated to the maximum length accepted by the model. The padding will be dealt with later on (in a data collator) so we pad examples to the longest length in the batch and not the whole dataset.

In [ ]:
prefix = "summarize: "


def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["article"]]  # Prefix for T5
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    targets = tokenizer(examples["highlights"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = targets["input_ids"]
    return model_inputs

In [ ]:
sample_dataset = Dataset.from_dict(dataset['train'][:2])
preprocessed_sample = preprocess_function(sample_dataset)

In [ ]:
preprocessed_sample

{'input_ids': [[21603, 10, 938, 3, 5, 24967, 3038, 588, 8888, 17, 3, 5, 3, 10744, 8775, 20619, 2326, 10, 3, 5, 12811, 10, 3628, 3, 6038, 6, 1003, 1186, 2038, 3, 5, 1820, 3, 5, 3, 6880, 4296, 11430, 10, 3, 5, 10668, 10, 2534, 3, 6038, 6, 1003, 1186, 2038, 3, 5, 37, 412, 5, 134, 5, 789, 65, 3, 9951, 223, 8, 7183, 21, 151, 16, 5053, 12, 1042, 70, 1104, 5146, 38, 8, 690, 16562, 12, 369, 12, 1353, 28, 2089, 31, 7, 18827, 6417, 6032, 5, 37, 7390, 16813, 7, 24, 646, 386, 3654, 11, 3, 24361, 7532, 44, 8, 5053, 19012, 808, 286, 30, 5287, 1430, 3, 18, 116, 928, 2055, 1104, 5146, 398, 36, 5132, 28, 8, 2822, 789, 5, 299, 8, 18524, 19764, 1387, 2162, 3, 9, 386, 18, 7393, 4924, 21, 15375, 7, 16, 5053, 16, 659, 13, 8, 3, 31, 449, 52, 2317, 18914, 31, 6, 2651, 24, 151, 16, 8, 690, 906, 72, 97, 5, 3, 31, 382, 21301, 2296, 18914, 31, 10, 37, 17615, 243, 151, 16, 5053, 113, 174, 72, 97, 12, 743, 70, 1104, 5146, 54, 43, 3, 9, 386, 18, 7393, 4924, 3, 31, 23016, 3516, 31, 826, 8, 18827, 984, 13, 2089, 3, 5,

To apply the preprocessing function over the entire dataset, use 🤗 Datasets [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map) method. You can speed up the `map` function by setting `batched=True` to process multiple elements of the dataset at once which will use multi-threading to treat the texts in a batch concurrently:

In [ ]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)

Now create a batch of examples using [DataCollatorForSeq2Seq](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorForSeq2Seq). It's more efficient to *dynamically pad* the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

data_collator is a function or class in Hugging Face Transformers that helps batch and pad inputs dynamically during training and evaluation. It ensures that all sequences in a batch have the same shape, handling tokenization inconsistencies automatically.

Handles Variable-Length Sequences:

Transformers require fixed-size input tensors, but text data varies in length.
The data_collator pads shorter sequences dynamically to match the longest sequence in a batch.

It improves memory efficiency and speeds up training.
Essential when training NLP models with Hugging Face Trainer API.

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
rouge

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

<Tip>

 Transformers provides a Trainer class optimized for training 🤗 Transformers models, making it easier to start training without manually writing your own training loop. The Trainer API supports a wide range of training options and features such as logging, gradient accumulation, and mixed precision.

</Tip>

You're ready to start training your model now! Since our task is of the sequence-to-sequence kind we Load T5 with [AutoModelForSeq2SeqLM](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForSeq2SeqLM):

In [ ]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

At this point, only three steps remain:

1. Define your training hyperparameters in [Seq2SeqTrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Seq2SeqTrainingArguments). The only required parameter is `output_dir` which specifies where to save your model. You'll push this model to the Hub by setting `push_to_hub=True` (you need to be signed in to Hugging Face to upload your model). At the end of each epoch, the [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) will evaluate the ROUGE metric and save the training checkpoint.
2. Pass the training arguments to [Seq2SeqTrainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Seq2SeqTrainer) along with the model, dataset, tokenizer, data collator, and `compute_metrics` function.
3. Call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

Here we set the evaluation to be done at the end of each epoch, tweak the learning rate, use the `batch_size` defined at the top of the cell and customize the weight decay. Since the `Seq2SeqTrainer` will save the model regularly and our dataset is quite large, we tell it to make three saves maximum. Lastly, we use the `predict_with_generate` option (to properly generate summaries) and activate mixed precision training (to go a bit faster)

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

- output_dir="./t5-small-finetuned-cnn-dailymail" → Directory to save model checkpoints and logs.
- evaluation_strategy="epoch" → Evaluate the model at the end of each epoch.
- learning_rate=2e-5 → Learning rate for the optimizer, controlling model updates.
- per_device_train_batch_size=16 → Number of training samples processed per batch on each device.
- per_device_eval_batch_size=16 → Number of evaluation samples processed per batch on each device.
- weight_decay=0.01 → Regularization term to prevent overfitting by penalizing large weights.
- num_train_epochs=2 → Number of times the model goes through the entire training dataset.
- save_strategy="epoch" → Save model checkpoints at the end of each epoch.
- fp16=True → Use mixed-precision (16-bit floating point) for faster training and lower memory usage on GPU.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="t5-small-finetuned-cnn-dailymail",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=True, #change to bf16=True for XPU
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,1.155500,1.043971,0.244600,0.111300,0.201200,0.201200,19.968800
2,1.144900,1.040560,0.244700,0.111400,0.201400,0.201500,19.968200


TrainOutput(global_step=17946, training_loss=1.159726337486853, metrics={'train_runtime': 11897.6046, 'train_samples_per_second': 24.132, 'train_steps_per_second': 1.508, 'total_flos': 3.885825530422886e+16, 'train_loss': 1.159726337486853, 'epoch': 2.0})

In [ ]:
# Validation
eval_results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])
print("Validation Results:")
print(eval_results)

# Testing (if you have a separate test dataset)
test_results = trainer.predict(test_dataset=tokenized_dataset["test"])
print("Test Results:")
print(test_results)

# If you want to print predictions from the test set:
predictions = test_results.predictions
labels = test_results.label_ids

# Decode predictions and labels for comparison (optional)
decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

# Print a few predictions and labels for inspection
for i in range(5):  # Print first 5 predictions
    print(f"Prediction: {decoded_preds[i]}")
    print(f"Ground Truth: {decoded_labels[i]}")
    print("-" * 50)


Push The Fine Tuned Model To the Hugging face Hub. You can view it on "https://huggingface.co/k200353/t5-small-finetuned-cnn-dailymail"

In [ ]:
from huggingface_hub import HfApi

# Define your Hugging Face model repo name (change 'your-username'!)
repo_name = "k200353/t5-small-finetuned-cnn-dailymail"

# Push model and tokenizer to Hugging Face Hub
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

## Pegasus

Pegasus model is **the best model for abstractive summary** which works on GSG (Gap Sentences), more details on the algorithim will be provided in presentation. The code below shows how fine tuning will be done for this model.

Model and Tokenizer Initialization

In [ ]:
import datasets
from transformers import PegasusTokenizer, PegasusForConditionalGeneration, Trainer, TrainingArguments

# 2. Load the Pegasus tokenizer and model pretrained on CNN/DailyMail
model_name = "google/pegasus-xsum"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenize the dataset

In [ ]:
# 3. Preprocessing: Tokenize the articles (inputs) and highlights (targets)
def preprocess_function(examples):
    inputs = tokenizer(examples["article"], max_length=512, truncation=True, padding="max_length")
    targets = tokenizer(text_target= examples["highlights"], max_length=128, truncation=True, padding="max_length")

    inputs["labels"] = targets["input_ids"]  # Assign target labels for training
    return inputs

In [ ]:
# Apply the preprocessing function over the entire dataset in batches.
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/114846 [00:00<?, ? examples/s]

In [ ]:
sample_dataset = Dataset.from_dict(dataset['train'][:2])
preprocessed_sample = preprocess_function(sample_dataset)

In [ ]:
preprocessed_sample

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

**Model Fine-Tuning**

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import torch
import gc

gc.collect()  # Collects Python garbage
torch.cuda.empty_cache()  # Clears GPU memory

Helps to clear out the GPU Cache for reruning

In [ ]:
!nvidia-smi

In [ ]:
model.to(device)  # Move model to GPU

# 4. Define training arguments. Adjust hyperparameters as needed.
training_args = TrainingArguments(
    output_dir="./pegasus-finetuned-cnn_dailymail",
    evaluation_strategy="epoch",
   learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=2,
    save_steps=1000,
    save_strategy="steps",
    fp16=True,  # Enable mixed precision for better performance
    # push_to_hub=True
     gradient_accumulation_steps = 16, #helps to run it on limited GPU Resources


)

# 5. Create a Trainer object with the model, training arguments, and datasets.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 6. Fine-tune the model
trainer.train(resume_from_checkpoint=True)

Pushing the Fine tuned model to HuggingFace Repository. Unfortunately, since the model required too high gpu resources, I was not able to publish this on my hugging face hub. (More Explanation will be provided during the presentation)

In [ ]:
from huggingface_hub import HfApi

repo_name = "k200353/pegasus-finetuned-cnn_dailymail"

# Push model and tokenizer to Hugging Face Hub
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

## BART


BART combines BERT’s bidirectional encoder with GPT-style autoregressive decoding. Its pretraining involves corrupting documents through several noise functions:

Create The Evaluation Metrics

In [ ]:
import evaluate

rouge = evaluate.load("rouge")
rouge

In [ ]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
import torch
from datasets import load_dataset, DatasetDict
from transformers import BartTokenizer, BartForConditionalGeneration, TrainingArguments, Trainer

Initialize the tokenizer and the Model

In [ ]:
# Load tokenizer and model
model_name = "facebook/bart-base"  # Smallest available BART model
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

Create the Preprocessor for the dataset

In [ ]:
# Function to preprocess dataset (tokenization)
def preprocess_function(examples):
    inputs = tokenizer(examples["article"], max_length=512, truncation=True, padding="max_length")
    targets = tokenizer(text_target= examples["highlights"], max_length=128, truncation=True, padding="max_length")

    inputs["labels"] = targets["input_ids"]  # Assign target labels for training
    return inputs



Apply The Preprocessing on the Entire Dataset

In [ ]:
# Apply preprocessing
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [ ]:
sample_dataset = Dataset.from_dict(dataset['train'][:2])
preprocessed_sample = preprocess_function(sample_dataset)

In [ ]:
preprocessed_sample

{'input_ids': [[0, 2765, 479, 9153, 1509, 3624, 119, 5229, 479, 29731, 7976, 14849, 1691, 35, 479, 15171, 35, 3305, 12936, 6, 601, 587, 1014, 479, 1721, 479, 121, 44964, 35, 479, 13470, 35, 1570, 12936, 6, 601, 587, 1014, 479, 20, 121, 4, 104, 4, 168, 34, 3148, 124, 5, 4267, 13, 82, 11, 2278, 7, 2870, 49, 629, 2886, 25, 5, 343, 6952, 7, 283, 7, 1110, 19, 302, 18, 7360, 4840, 1912, 4, 20, 9544, 18740, 14, 314, 130, 1462, 8, 29568, 1710, 23, 5, 2278, 13518, 362, 317, 15, 6394, 1053, 111, 77, 1736, 1425, 629, 2886, 531, 28, 1658, 19, 5, 752, 168, 4, 125, 5, 18387, 5833, 1841, 585, 10, 130, 12, 2151, 5064, 13, 7660, 11, 2278, 11, 1109, 9, 5, 128, 1334, 24786, 6906, 3934, 1271, 14, 82, 11, 5, 343, 956, 55, 86, 4, 128, 32579, 24786, 6906, 13373, 20, 12439, 26, 82, 11, 2278, 54, 240, 55, 86, 7, 1498, 49, 629, 2886, 64, 33, 10, 130, 12, 2151, 5064, 128, 19010, 4022, 108, 511, 5, 7360, 1061, 9, 302, 479, 20, 2278, 13518, 16, 547, 15, 4314, 108, 1053, 6, 5, 371, 302, 11, 587, 6, 15, 61, 5517, 50

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

**Prepare For Model Fine Tuning**

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
model.to(device)  # Move model to GPU

# Define training arguments optimized for CPU
training_args = TrainingArguments(
    output_dir="./bart-finetuned-cnn-dailymail",
    eval_strategy="epoch",       # Optionally change evaluation frequency as well
    save_strategy="steps",             # Save checkpoints at regular step intervals
    save_steps=1000,                    # Save every 500 steps (adjust as needed)
    learning_rate=2e-3,  # Lower LR for stable training on CPU
    per_device_train_batch_size=1,  # Reduce batch size due to CPU memory limits
    per_device_eval_batch_size=1,
    gradient_accumulation_steps = 8,
    num_train_epochs=2,  # Reduce epochs to speed up training
    weight_decay=0.01,
    save_total_limit=2,  # Keep only the last 2 checkpoints
    fp16=True,
    eval_accumulation_steps=4  #When computing metrics inside the Trainer, your predictions are all gathered together on the device (GPU/TPU) and only passed back to the CPU at the end (because that operation can be slow). If your dataset is large (or your model outputs large predictions) you can use eval_accumulation_steps to set a number of steps after which your predictions are sent back to the CPU (slower but uses less device memory). This should avoid your OOM.
    #Didnt Work
)


# Trainer object
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Fine-tune the model
trainer.train(resume_from_checkpoint=True)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: k200353 (fast-k200353) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Epoch,Training Loss,Validation Loss


Once done publish the model to the Hugging Face.

In [ ]:
from huggingface_hub import HfApi

# Define your Hugging Face model repo name (change 'your-username'!)
repo_name = "k200353/pegasus-finetuned-cnn_dailymail"

# Push model and tokenizer to Hugging Face Hub
model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

## Comparitative Analysis For Each Of The Pretrained models

In [ ]:
from transformers import pipeline
from rouge_score import rouge_scorer

# Define the models to use
models = {
    "facebook/bart-base": pipeline("summarization", model="facebook/bart-base"),
    "t5-small": pipeline("summarization", model="t5-small"),
    "google/pegasus-xsum": pipeline("summarization", model="google/pegasus-xsum"),
}

# Sample text for summarization
document = """
    In 1969, NASA successfully landed the first humans on the Moon as part of the Apollo 11 mission.
    Astronauts Neil Armstrong and Buzz Aldrin spent about 21 hours on the lunar surface, collecting samples
    and conducting experiments. This historic event marked a significant milestone in space exploration,
    inspiring generations of scientists and engineers. The Apollo program continued with subsequent missions,
    expanding our understanding of the Moon and paving the way for future space travel. Today, agencies like NASA
    and private companies are working towards returning humans to the Moon through the Artemis program, with the
    goal of establishing a sustainable presence and preparing for future missions to Mars.
"""

# Generate summaries
summaries = {}
for model_name, model_pipeline in models.items():
    summary = model_pipeline(document, max_length=100, min_length=10, do_sample=False)[0]["summary_text"]
    summaries[model_name] = summary

# Print summaries
for model_name, summary in summaries.items():
    print(f"{model_name} Summary:")
    print(summary)
    print()

# Compute ROUGE scores
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

results = {}
for model_name, summary in summaries.items():
    scores = scorer.score(document, summary)
    results[model_name] = {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure,
    }

# Print results
for model_name, scores in results.items():
    print(f"{model_name} ROUGE Scores:")
    for rouge_type, score in scores.items():
        print(f"  {rouge_type}: {score:.4f}")
    print()


**Facebook/bart-base Summary:**
NASA    In 1969, NASA successfully landed the first humans on the Moon as part of the Apollo 11 mission. Â   Astronauts Neil Armstrong and Buzz Aldrin spent about 21 hours on the lunar surface, collecting samples   ,   and conducting experiments. This historic event marked a significant milestone in space exploration,     inspiring generations of scientists and engineers. The Apollo program continued with subsequent missions,   including the first manned mission to the Moon in

**t5-small Summary:**
in 1969, NASA successfully landed the first humans on the moon as part of the Apollo 11 mission . astronauts spent about 21 hours on the lunar surface collecting samples and conducting experiments . this historic event marked a significant milestone in space exploration .

**google/pegasus-xsum Summary:**
This week marks the 50th anniversary of the first humans to set foot on the Moon.

- **facebook/bart-base:**

Achieved the highest ROUGE scores (ROUGE-1: 0.7416, ROUGE-2: 0.7045, ROUGE-L: 0.7191).
Its summary closely mirrors the key content of the original text, even though it appears slightly truncated

- **t5-small**

performed mediocre (ROUGE-1: 0.5600, ROUGE-2: 0.5405, ROUGE-L: 0.5600), with a summary that, while concise, missed some details.

- **google/pegasus-xsum:**

Recorded the lowest scores (ROUGE-1: 0.1774, ROUGE-2: 0.0984, ROUGE-L: 0.1452).
Its summary diverged from the main content by focusing on an anniversary angle that wasn’t present in the input, resulting in minimal overlap with the reference summary.

However in terms of abstractive summarization and not extractive summarization Google Pegaus Xsum performed the best as it was the only model which was able to paraphrase

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from rouge_score import rouge_scorer

# List of models to evaluate
model_choices = ["t5-small", "t5-base", "k200353/t5-small-finetuned-cnn-dailymail", "facebook/bart-base", "google/pegasus-xsum"]

# Load CNN/DailyMail dataset (subset for efficiency)
dataset = load_dataset("cnn_dailymail", "3.0.0", split="test[:10]")

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

results = {}

for model_choice in model_choices:
    # Load the tokenizer and model for the current choice
    tokenizer = AutoTokenizer.from_pretrained(model_choice)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_choice)

    rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for sample in dataset:
        document = sample["article"]
        reference_summary = sample["highlights"]

        tokens = tokenizer(document, truncation=True, padding="longest", return_tensors="pt")
        summary = model.generate(**tokens)

        # Generate summary using conditional logic
        if model_choice in ["t5-small", "t5-base"]:
            # Note the prompt typo 'sumarize:' as provided
            tokens = tokenizer.encode("sumarize: " + document, return_tensors='pt', padding="longest", truncation=True)
            summary = model.generate(tokens)
        if model_choice == "k200353/t5-small-finetuned-cnn-dailymail":
            inputs = tokenizer(document, return_tensors="pt", max_length=512, truncation=True, padding="longest")
            summary = model.generate(inputs['input_ids'], max_length=150, num_beams=4, early_stopping=True)

        decoded_output = tokenizer.decode(summary[0], skip_special_tokens=True)

        # Compute ROUGE scores comparing the reference summary to the generated summary
        scores = scorer.score(reference_summary, decoded_output)
        rouge_scores["rouge1"].append(scores["rouge1"].fmeasure)
        rouge_scores["rouge2"].append(scores["rouge2"].fmeasure)
        rouge_scores["rougeL"].append(scores["rougeL"].fmeasure)

    results[model_choice] = {
        "ROUGE-1": sum(rouge_scores["rouge1"]) / len(rouge_scores["rouge1"]),
        "ROUGE-2": sum(rouge_scores["rouge2"]) / len(rouge_scores["rouge2"]),
        "ROUGE-L": sum(rouge_scores["rougeL"]) / len(rouge_scores["rougeL"]),
    }

# Print results
for model_choice, scores in results.items():
    print(f"{model_choice} ROUGE Scores on CNN/DailyMail:")
    for rouge_type, score in scores.items():
        print(f"  {rouge_type}: {score:.4f}")
    print()


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


t5-small ROUGE Scores on CNN/DailyMail:
  ROUGE-1: 0.2250
  ROUGE-2: 0.1055
  ROUGE-L: 0.1724

t5-base ROUGE Scores on CNN/DailyMail:
  ROUGE-1: 0.2788
  ROUGE-2: 0.1032
  ROUGE-L: 0.2005

k200353/t5-small-finetuned-cnn-dailymail ROUGE Scores on CNN/DailyMail:
  ROUGE-1: 0.3045
  ROUGE-2: 0.0870
  ROUGE-L: 0.2152

facebook/bart-base ROUGE Scores on CNN/DailyMail:
  ROUGE-1: 0.1740
  ROUGE-2: 0.0643
  ROUGE-L: 0.1586

google/pegasus-xsum ROUGE Scores on CNN/DailyMail:
  ROUGE-1: 0.2617
  ROUGE-2: 0.0913
  ROUGE-L: 0.1934



-   **k200353/t5-small-finetuned-cnn-dailymail**
    
    -   **ROUGE-1:** 0.3045
    -   **ROUGE-2:** 0.0870
    -   **ROUGE-L:** 0.2152
    -   _Observation:_ This model, being fine-tuned specifically on CNN/DailyMail data, outperforms the others in terms of overall content overlap (ROUGE-1) and longest common subsequence (ROUGE-L), suggesting that it captures the essence of the reference summaries better.
-   **t5-base**
    
    -   **ROUGE-1:** 0.2788
    -   **ROUGE-2:** 0.1032
    -   **ROUGE-L:** 0.2005
    -   _Observation:_ t5-base shows strong performance, especially in ROUGE-1 and ROUGE-L. Its ROUGE-2 score is similar to that of t5-small, indicating comparable bigram overlaps, but its overall performance benefits from the larger model capacity.
-   **google/pegasus-xsum**
    
    -   **ROUGE-1:** 0.2617
    -   **ROUGE-2:** 0.0913
    -   **ROUGE-L:** 0.1934
    -   _Observation:_ Pegasus-xsum performs moderately, falling between the t5 models in ROUGE-1 and ROUGE-L. It is less competitive compared to the fine-tuned model on this particular dataset.
-   **t5-small**
    
    -   **ROUGE-1:** 0.2250
    -   **ROUGE-2:** 0.1055
    -   **ROUGE-L:** 0.1724
    -   _Observation:_ While t5-small scores slightly lower on ROUGE-1 and ROUGE-L compared to t5-base, its ROUGE-2 is marginally higher. This suggests that although it may capture some local (bigram) overlaps well, it might miss some broader content coverage.
-   **facebook/bart-base**
    
    -   **ROUGE-1:** 0.1740
    -   **ROUGE-2:** 0.0643
    -   **ROUGE-L:** 0.1586
    -   _Observation:_ Bart-base lags behind the others across all metrics, which might indicate that it is less suited for this dataset without additional fine-tuning or adaptation.

### Summary of Observations

-   **Fine-tuning Matters:** The fine-tuned model (k200353/t5-small-finetuned-cnn-dailymail) clearly outperforms the other models, highlighting the benefit of task-specific training.
-   **Model Capacity vs. Specialization:** t5-base benefits from its larger capacity, leading to better ROUGE-1 and ROUGE-L scores compared to t5-small, even though t5-small shows a slight edge in ROUGE-2.
-   **Architecture Differences:** While google/pegasus-xsum and facebook/bart-base are strong models in many scenarios, their performance here suggests they may not be as effective as the T5 variants on the CNN/DailyMail dataset, potentially due to differences in training data and fine-tuning.
-   **Practical Implications:** For tasks like summarization on datasets similar to CNN/DailyMail, using a model that has been fine-tuned on that specific domain (or closely related data) can yield significantly better results.